<center><h1 style="margin: 1.5em 0;">Mathematical Explanation of the MNISTNet Implementation</h1></center>



A 3-layer fully connected neural network (784 → 256 → 128 → 10) trained with **mini-batch gradient descent**, **ReLU** hidden activations, **softmax** output, **cross-entropy loss** + **L2 regularization**.

## 1. Notation

- $X \in \mathbb{R}^{m \times 784}$ : mini-batch of flattened images  
- $Y \in \mathbb{R}^{m \times 10}$ : one-hot encoded true labels  
- $m$ : batch size  
- $W^{(l)} \in \mathbb{R}^{n_l \times n_{l-1}}$ : weight matrix of layer $l$  
- $b^{(l)} \in \mathbb{R}^{1 \times n_l}$ : bias vector  
- $Z^{(l)}, A^{(l)} \in \mathbb{R}^{m \times n_l}$ : pre- and post-activation of layer $l$  
- $A^{(0)} := X$

Layers:  
- $l=1$: 784 → 256  
- $l=2$: 256 → 128  
- $l=3$: 128 → 10 (output)

## 2. Forward Pass

$$
\begin{aligned}
Z^{(1)} &= X W^{(1)\top} + b^{(1)} \\
A^{(1)} &= \operatorname{ReLU}(Z^{(1)}) && \text{where } \operatorname{ReLU}(z) = \max(0,z) \\[1em]
Z^{(2)} &= A^{(1)} W^{(2)\top} + b^{(2)} \\
A^{(2)} &= \operatorname{ReLU}(Z^{(2)}) \\[1em]
Z^{(3)} &= A^{(2)} W^{(3)\top} + b^{(3)} \\
A^{(3)} &= \operatorname{softmax}(Z^{(3)})
\end{aligned}
$$

**Stable softmax** (used in code):

$$
\operatorname{softmax}(z_i) = \frac{\exp(z_i - \max_j z_j)}{\sum_k \exp(z_k - \max_j z_j)}
$$

$A^{(3)}$ contains predicted probabilities $\hat{y}_i \in [0,1]^{10}$ with $\sum_c \hat{y}_{i,c} = 1$.

## 3. Loss Function

**Per mini-batch loss** (average cross-entropy + L2 regularization):

$$
\mathcal{L} = -\frac{1}{m} \sum_{i=1}^m \sum_{c=1}^{10} y_{i,c} \log(\hat{y}_{i,c}) + \frac{\lambda}{2m} \sum_{l=1}^3 \|W^{(l)}\|_F^2
$$

- First term = categorical cross-entropy (CE)  
- Second term = weight decay (L2 regularization)  
- $\lambda$ = regularization strength (`reg` in code)

## 4. Backpropagation – Gradient Computation

### Output layer (l = 3)

$$
\frac{\partial \mathcal{L}}{\partial Z^{(3)}} = A^{(3)} - Y \qquad \text{(very convenient property of softmax + CE)}
$$

$$
\frac{\partial \mathcal{L}}{\partial W^{(3)}} = \frac{1}{m} \left( {dZ^{(3)}}^\top A^{(2)} \right) + \frac{\lambda}{m} W^{(3)}
$$

$$
\frac{\partial \mathcal{L}}{\partial b^{(3)}} = \frac{1}{m} \sum_{i=1}^m dZ^{(3)}_{i,:} \quad \text{(row vector)}
$$

### Hidden layers (l = 2 and l = 1)

$$
dZ^{(l)} = \left( dZ^{(l+1)} W^{(l+1)} \right) \odot \mathbb{1}_{\{Z^{(l)} > 0\}}
$$

$$
\frac{\partial \mathcal{L}}{\partial W^{(l)}} = \frac{1}{m} \left( {dZ^{(l)}}^\top A^{(l-1)} \right) + \frac{\lambda}{m} W^{(l)}
$$

$$
\frac{\partial \mathcal{L}}{\partial b^{(l)}} = \frac{1}{m} \sum_{i=1}^m dZ^{(l)}_{i,:}
$$

(where $A^{(0)} = X$ and $\odot$ is element-wise multiplication)

## 5. Parameter Update (Vanilla Gradient Descent)

$$
W^{(l)} \leftarrow W^{(l)} - \eta \cdot \frac{\partial \mathcal{L}}{\partial W^{(l)}}, \quad
b^{(l)} \leftarrow b^{(l)} - \eta \cdot \frac{\partial \mathcal{L}}{\partial b^{(l)}}
$$

$\eta =$ learning rate (`lr` in code)

## 6. Weight Initialization – He initialization (for ReLU)

$$
W^{(l)}_{ij} \sim \mathcal{N}\left(0, \sqrt{\frac{2}{n_{l-1}}}\right)
$$

This scaling helps keep the variance of activations roughly constant across layers when using ReLU.

## Summary Table

| Layer | Input dim | Output dim | Activation | Gradient w.r.t. Z          |
|-------|-----------|------------|------------|-----------------------------|
| 1     | 784       | 256        | ReLU       | $(dZ^{(2)} W^{(2)}) \odot \mathbb{1}_{Z^{(1)}>0}$ |
| 2     | 256       | 128        | ReLU       | $(dZ^{(3)} W^{(3)}) \odot \mathbb{1}_{Z^{(2)}>0}$ |
| 3     | 128       | 10         | softmax    | $A^{(3)} - Y$               |



In [1]:
import numpy as np
from tensorflow.keras.datasets import mnist

In [2]:
class MNISTNet:
    def __init__(self, lr=0.05, reg=1e-4):
        self.lr = lr
        self.reg = reg
        self._init_params()

    def _init_params(self):
        dims = [784, 256, 128, 10]
        self.params = {}

        for i in range(1, len(dims)):
            self.params[f"W{i}"] = (
                np.random.randn(dims[i], dims[i-1]) *
                np.sqrt(2.0 / dims[i-1])
            )
            self.params[f"b{i}"] = np.zeros((1, dims[i]))  # row bias

    # --------------------------------------------------
    # ACTIVATIONS
    # --------------------------------------------------

    def relu(self, Z):
        return np.maximum(0, Z)

    def relu_grad(self, Z):
        return (Z > 0).astype(float)

    def softmax(self, Z):
        Z_stable = Z - np.max(Z, axis=1, keepdims=True)
        exp = np.exp(Z_stable)
        return exp / np.sum(exp, axis=1, keepdims=True)

    # --------------------------------------------------
    # FORWARD
    # --------------------------------------------------

    def forward(self, X):
        self.cache = {}

        W1, b1 = self.params["W1"], self.params["b1"]
        W2, b2 = self.params["W2"], self.params["b2"]
        W3, b3 = self.params["W3"], self.params["b3"]

        # (m,784)(784,256) = (m,256)
        Z1 = X @ W1.T + b1
        A1 = self.relu(Z1)

        Z2 = A1 @ W2.T + b2
        A2 = self.relu(Z2)

        Z3 = A2 @ W3.T + b3
        A3 = self.softmax(Z3)

        self.cache = {
            "X": X,
            "Z1": Z1, "A1": A1,
            "Z2": Z2, "A2": A2,
            "Z3": Z3, "A3": A3
        }

        return A3

    # --------------------------------------------------
    # LOSS
    # --------------------------------------------------

    def compute_loss(self, Y_hat, Y):
        m = Y.shape[0]

        ce = -np.sum(Y * np.log(Y_hat + 1e-8)) / m

        l2 = sum(np.sum(self.params[f"W{i}"]**2) for i in range(1,4))
        l2 = (self.reg / (2*m)) * l2

        return ce + l2

    # --------------------------------------------------
    # BACKPROP
    # --------------------------------------------------

    def backward(self, Y):
        m = Y.shape[0]
        grads = {}

        X  = self.cache["X"]
        A1 = self.cache["A1"]
        A2 = self.cache["A2"]
        A3 = self.cache["A3"]
        Z1 = self.cache["Z1"]
        Z2 = self.cache["Z2"]

        W1 = self.params["W1"]
        W2 = self.params["W2"]
        W3 = self.params["W3"]

        # (m,10)
        dZ3 = A3 - Y

        grads["W3"] = (dZ3.T @ A2) / m + (self.reg/m)*W3
        grads["b3"] = np.sum(dZ3, axis=0, keepdims=True) / m

        dZ2 = (dZ3 @ W3) * self.relu_grad(Z2)

        grads["W2"] = (dZ2.T @ A1) / m + (self.reg/m)*W2
        grads["b2"] = np.sum(dZ2, axis=0, keepdims=True) / m

        dZ1 = (dZ2 @ W2) * self.relu_grad(Z1)

        grads["W1"] = (dZ1.T @ X) / m + (self.reg/m)*W1
        grads["b1"] = np.sum(dZ1, axis=0, keepdims=True) / m

        return grads

    # --------------------------------------------------
    # UPDATE
    # --------------------------------------------------

    def update(self, grads):
        for i in range(1,4):
            self.params[f"W{i}"] -= self.lr * grads[f"W{i}"]
            self.params[f"b{i}"] -= self.lr * grads[f"b{i}"]

    # --------------------------------------------------
    # TRAIN
    # --------------------------------------------------

    def train(self, X_train, Y_train, X_test, Y_test,
              epochs=20, batch_size=128):

        m = X_train.shape[0]

        for epoch in range(1, epochs+1):
            idx = np.random.permutation(m)

            for i in range(0, m, batch_size):
                batch_idx = idx[i:i+batch_size]

                X_batch = X_train[batch_idx]
                Y_batch = Y_train[batch_idx]

                Y_hat = self.forward(X_batch)
                grads = self.backward(Y_batch)
                self.update(grads)

            train_acc = self.accuracy(X_train, Y_train)
            test_acc  = self.accuracy(X_test, Y_test)
            loss = self.compute_loss(self.forward(X_train), Y_train)

            print(f"Epoch {epoch:2d} | "
                  f"Loss {loss:.4f} | "
                  f"Train {train_acc:.2f}% | "
                  f"Test {test_acc:.2f}%")

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

    def accuracy(self, X, Y):
        preds = self.predict(X)
        labels = np.argmax(Y, axis=1)
        return np.mean(preds == labels) * 100

In [3]:
import numpy as np
from tensorflow.keras.datasets import mnist

print("Loading MNIST...")
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

# Flatten images (keep 60000, 784 format)
X_train = X_train.reshape(60000, -1) / 255.0
X_test  = X_test.reshape(10000, -1)  / 255.0

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

Loading MNIST...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train shape: (60000, 784)
Test shape : (10000, 784)


In [4]:
def one_hot(Y, num_classes=10):
    return np.eye(num_classes)[Y]

Y_train_oh = one_hot(Y_train)
Y_test_oh  = one_hot(Y_test)

print("Y_train shape:", Y_train_oh.shape)

Y_train shape: (60000, 10)


In [5]:
net = MNISTNet(lr=0.05, reg=1e-4)

net.train(
    X_train, Y_train_oh,
    X_test,  Y_test_oh,
    epochs=20,
    batch_size=128
)

Epoch  1 | Loss 0.3018 | Train 91.27% | Test 91.65%
Epoch  2 | Loss 0.2078 | Train 94.19% | Test 94.08%
Epoch  3 | Loss 0.1675 | Train 95.20% | Test 94.96%
Epoch  4 | Loss 0.1429 | Train 96.00% | Test 95.60%
Epoch  5 | Loss 0.1234 | Train 96.54% | Test 96.03%
Epoch  6 | Loss 0.1081 | Train 96.98% | Test 96.59%
Epoch  7 | Loss 0.0975 | Train 97.26% | Test 96.76%
Epoch  8 | Loss 0.0885 | Train 97.52% | Test 96.89%
Epoch  9 | Loss 0.0796 | Train 97.84% | Test 97.00%
Epoch 10 | Loss 0.0738 | Train 97.88% | Test 97.09%
Epoch 11 | Loss 0.0642 | Train 98.29% | Test 97.32%
Epoch 12 | Loss 0.0602 | Train 98.31% | Test 97.29%
Epoch 13 | Loss 0.0576 | Train 98.41% | Test 97.27%
Epoch 14 | Loss 0.0497 | Train 98.69% | Test 97.54%
Epoch 15 | Loss 0.0506 | Train 98.61% | Test 97.36%
Epoch 16 | Loss 0.0427 | Train 98.92% | Test 97.62%
Epoch 17 | Loss 0.0406 | Train 98.97% | Test 97.61%
Epoch 18 | Loss 0.0385 | Train 99.04% | Test 97.73%
Epoch 19 | Loss 0.0333 | Train 99.21% | Test 97.70%
Epoch 20 | L

In [6]:
"""
A neural network that recognizes handwritten digits (0-9), built from
scratch with numpy -- no deep learning frameworks.

Notation used throughout:
    m   = number of images in a batch (e.g. 128)
    784 = pixels per image (28 x 28 flattened into one row)
    W   = weight matrix, b = bias vector
    Z   = raw weighted sum (before activation)
    A   = activated output (after ReLU or softmax)

Shape convention: every batch of data is a matrix of shape (m, features).
Each ROW is one image. This is the standard convention (as opposed to
each image being a column) and it's why some formulas below use W.T
(transpose) to make the matrix multiplication line up.
"""

import numpy as np


# ======================================================================
# 1. LOAD THE DATA
# ======================================================================

def load_mnist():
    """
    Downloads MNIST (70,000 images total) and splits it the standard way:
    60,000 for training, 10,000 for testing.

    Raw shapes coming out of fetch_openml:
        X: (70000, 784)  -- each row is one flattened 28x28 image, values 0-255
        y: (70000,)       -- each entry is a single digit label, 0-9
    """
    from sklearn.datasets import fetch_openml

    print("Downloading MNIST (only happens once, then it's cached)...")
    mnist = fetch_openml('mnist_784', version=1, as_frame=False)
    X, y = mnist.data, mnist.target.astype(int)

    X_train, X_test = X[:60000], X[60000:]
    y_train, y_test = y[:60000], y[60000:]

    return X_train, y_train, X_test, y_test


def preprocess(X, y, num_classes=10):
    """
    Two transformations happen here:

    1. Pixel scaling: raw pixels are 0-255. Neural nets train much more
       reliably when inputs are small numbers, so we divide by 255 to
       squeeze everything into the range [0, 1].
       Shape unchanged: (m, 784) -> (m, 784)

    2. One-hot encoding: a label like 7 becomes a row vector
       [0,0,0,0,0,0,0,1,0,0]  (a 1 in position 7, zeros elsewhere).
       This is the format the loss function and backprop math below
       expect -- it lets us compare "probability of each digit" directly
       against "truth for each digit".
       Shape: (m,) -> (m, 10)
    """
    X_scaled = X / 255.0
    Y_onehot = np.eye(num_classes)[y]
    return X_scaled, Y_onehot


# ======================================================================
# 2. THE NEURAL NETWORK
# ======================================================================

class SimpleMNISTNet:
    """
    Architecture: 784 -> 256 -> 128 -> 10

    Picture it as 4 layers of numbers, connected by 3 sets of weights:

        Input        Hidden 1      Hidden 2      Output
        (784)   -W1->  (256)  -W2->  (128)  -W3->  (10)
                ReLU           ReLU           Softmax

    Every arrow is a full matrix multiplication -- every neuron in one
    layer is connected to every neuron in the next.
    """

    def __init__(self, learning_rate=0.05, l2_strength=1e-4):
        self.lr = learning_rate     # step size for gradient descent
        self.reg = l2_strength      # penalty on large weights (prevents overfitting)
        self.layer_sizes = [784, 256, 128, 10]
        self.num_layers = len(self.layer_sizes) - 1   # = 3 sets of weights
        self._init_weights()

    def _init_weights(self):
        """
        For each of the 3 connections, create:
            W with shape (fan_out, fan_in)
            b with shape (1, fan_out)

        Example for W1 (input -> hidden 1):
            fan_in = 784, fan_out = 256  ->  W1.shape = (256, 784)
            That's 256 neurons, each with 784 incoming weights.

        Why random, and why this particular scale?
            - Starting all weights at the same value (e.g. 0) means every
              neuron computes the identical thing and learns identically
              forever -- the network would never differentiate.
            - Scaling by sqrt(2 / fan_in) ("He initialization") keeps the
              variance of the signal roughly constant as it passes through
              layers, so values don't explode or shrink to zero.
        """
        self.weights = []
        self.biases = []

        for i in range(self.num_layers):
            fan_in = self.layer_sizes[i]
            fan_out = self.layer_sizes[i + 1]

            W = np.random.randn(fan_out, fan_in) * np.sqrt(2.0 / fan_in)
            b = np.zeros((1, fan_out))

            self.weights.append(W)
            self.biases.append(b)

    # ------------------------------------------------------------
    # ACTIVATION FUNCTIONS
    # ------------------------------------------------------------

    def relu(self, Z):
        """
        ReLU(z) = max(0, z)

        Without an activation function, stacking 3 linear layers would
        be mathematically equivalent to just 1 linear layer -- you need
        a non-linearity for the network to learn curved decision
        boundaries instead of straight lines.
        Shape: unchanged, e.g. (m, 256) -> (m, 256)
        """
        return np.maximum(0, Z)

    def relu_derivative(self, Z):
        """
        The slope of ReLU is 1 wherever z > 0, and 0 wherever z <= 0.
        Needed during backprop to know how much a small change in Z
        affects the output A -- that's what a derivative measures.
        Shape: unchanged, e.g. (m, 256) -> (m, 256)
        """
        return (Z > 0).astype(float)

    def softmax(self, Z):
        """
        softmax(z)_k = exp(z_k) / sum_j( exp(z_j) )

        Converts 10 raw scores into 10 probabilities that sum to 1.
        We subtract the row max first purely for numerical safety --
        exp() of a large number overflows, and subtracting a constant
        from every entry in a row doesn't change the final softmax
        result (it cancels out in the division).
        Shape: (m, 10) -> (m, 10)
        """
        Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
        exp_scores = np.exp(Z_shifted)
        return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

    # ------------------------------------------------------------
    # FORWARD PASS: image in, prediction out
    # ------------------------------------------------------------

    def forward(self, X):
        """
        X has shape (m, 784) -- m images, 784 pixels each.

        For each layer:  Z = A_prev @ W.T + b
            A_prev: (m, fan_in)
            W.T:    (fan_in, fan_out)
            result: (m, fan_out)
            + b (1, fan_out) broadcasts across all m rows

        Concretely, for layer 1:
            A0 (m, 784) @ W1.T (784, 256)  ->  Z1 (m, 256)
            Z1 -> ReLU -> A1 (m, 256)

        layer 2:
            A1 (m, 256) @ W2.T (256, 128)  ->  Z2 (m, 128)
            Z2 -> ReLU -> A2 (m, 128)

        layer 3 (output):
            A2 (m, 128) @ W3.T (128, 10)   ->  Z3 (m, 10)
            Z3 -> softmax -> A3 (m, 10)     <- final prediction

        Every Z and A gets stored in self.cache because backward()
        needs them to compute gradients -- you can't know how much a
        layer contributed to the error without knowing what it output.
        """
        self.cache = {"A0": X}
        A = X

        for i in range(self.num_layers):
            W, b = self.weights[i], self.biases[i]
            Z = A @ W.T + b
            is_last_layer = (i == self.num_layers - 1)
            A = self.softmax(Z) if is_last_layer else self.relu(Z)

            self.cache[f"Z{i+1}"] = Z
            self.cache[f"A{i+1}"] = A

        return A   # shape (m, 10): predicted probability of each digit

    # ------------------------------------------------------------
    # LOSS: how wrong was the prediction?
    # ------------------------------------------------------------

    def compute_loss(self, predictions, targets):
        """
        predictions: (m, 10) probabilities from forward()
        targets:     (m, 10) one-hot true labels

        Cross-entropy loss:
            loss = -(1/m) * sum( targets * log(predictions) )

        Because targets is one-hot (only one 1 per row), this reduces to
        just -log(predicted probability of the correct digit), averaged
        over the batch. A confident correct answer (p=0.99) gives a tiny
        loss; a confident wrong answer (p=0.01 for the true class) gives
        a huge loss. This is what makes the network punish confident
        mistakes much more than uncertain ones.

        The + 1e-8 avoids log(0), which is undefined.

        L2 regularization term:
            (reg / 2m) * sum(W^2) for every weight matrix

        This adds a small cost proportional to the size of the weights,
        which discourages the network from relying on a few huge
        weights (a common sign of overfitting to the training data).

        Returns a single number (the loss for this batch).
        """
        m = targets.shape[0]

        data_loss = -np.sum(targets * np.log(predictions + 1e-8)) / m

        weight_penalty = sum(np.sum(W ** 2) for W in self.weights)
        l2_loss = (self.reg / (2 * m)) * weight_penalty

        return data_loss + l2_loss

    # ------------------------------------------------------------
    # BACKWARD PASS: figure out how to fix the mistake
    # ------------------------------------------------------------

    def backward(self, targets):
        """
        targets: (m, 10) one-hot true labels.

        STEP 1 -- error at the output layer.
        For softmax combined with cross-entropy loss, the derivative of
        the loss with respect to Z3 simplifies beautifully to:
            dZ3 = A3 - targets              shape (m, 10)
        (predicted probabilities minus the true one-hot vector)

        STEP 2 -- gradient for this layer's weights and biases.
        For any layer i, once you have dZ (the error at that layer's
        pre-activation), the gradients are:
            dW_i = (dZ.T @ A_prev) / m + (reg/m) * W_i
            db_i = sum(dZ, over batch) / m

        Dimension check for layer 3:
            dZ3.T:  (10, m)
            A2:     (m, 128)
            dZ3.T @ A2 -> (10, 128)   <- exactly W3's shape. Good.

        STEP 3 -- push the error back to the previous layer.
            dZ_prev = (dZ @ W) * relu_derivative(Z_prev)

        Dimension check going from layer 3 back to layer 2:
            dZ3:  (m, 10)
            W3:   (10, 128)
            dZ3 @ W3 -> (m, 128)      <- matches Z2's shape
            multiplied elementwise by relu_derivative(Z2), which is
            also (m, 128) -- this zeroes out the error for any neuron
            that was already off (output 0) during the forward pass,
            since an inactive neuron couldn't have contributed to the
            output and shouldn't be blamed for the error.

        This repeats for each layer, walking backward from output to
        input -- hence "backpropagation".
        """
        m = targets.shape[0]
        grads_W = [None] * self.num_layers
        grads_b = [None] * self.num_layers

        dZ = self.cache[f"A{self.num_layers}"] - targets   # (m, 10)

        for i in reversed(range(self.num_layers)):
            A_prev = self.cache[f"A{i}"]
            W = self.weights[i]

            grads_W[i] = (dZ.T @ A_prev) / m + (self.reg / m) * W
            grads_b[i] = np.sum(dZ, axis=0, keepdims=True) / m

            if i > 0:
                Z_prev = self.cache[f"Z{i}"]
                dZ = (dZ @ W) * self.relu_derivative(Z_prev)

        return grads_W, grads_b

    # ------------------------------------------------------------
    # UPDATE: nudge the weights to reduce the error
    # ------------------------------------------------------------

    def update(self, grads_W, grads_b):
        """
        Gradient descent: move every weight a small step in the
        direction that reduces the loss.

            W = W - learning_rate * dW

        The gradient points in the direction of steepest INCREASE in
        loss, so we subtract it to go downhill instead. learning_rate
        controls the step size -- too big and training overshoots and
        diverges, too small and training crawls.
        """
        for i in range(self.num_layers):
            self.weights[i] -= self.lr * grads_W[i]
            self.biases[i] -= self.lr * grads_b[i]

    # ------------------------------------------------------------
    # TRAINING LOOP
    # ------------------------------------------------------------

    def train(self, X_train, Y_train, X_test, Y_test, epochs=20, batch_size=128):
        """
        One epoch = one full pass over all training images, done in
        small batches of `batch_size` images at a time (instead of all
        60,000 at once, which is slow and memory-heavy, or one at a
        time, which is noisy and slow to converge).

        For each batch: forward -> measure loss -> backward -> update.
        """
        m = X_train.shape[0]

        for epoch in range(1, epochs + 1):
            shuffled_idx = np.random.permutation(m)

            for start in range(0, m, batch_size):
                batch_idx = shuffled_idx[start:start + batch_size]
                X_batch = X_train[batch_idx]
                Y_batch = Y_train[batch_idx]

                self.forward(X_batch)
                grads_W, grads_b = self.backward(Y_batch)
                self.update(grads_W, grads_b)

            train_acc = self.accuracy(X_train, Y_train)
            test_acc = self.accuracy(X_test, Y_test)
            loss = self.compute_loss(self.forward(X_train), Y_train)

            print(f"Epoch {epoch:2d} | Loss {loss:.4f} | "
                  f"Train {train_acc:.2f}% | Test {test_acc:.2f}%")

    def predict(self, X):
        """
        forward(X) gives (m, 10) probabilities. argmax picks the index
        (0-9) with the highest probability in each row -- that's the
        network's guessed digit.
        """
        return np.argmax(self.forward(X), axis=1)

    def accuracy(self, X, Y):
        predictions = self.predict(X)
        true_labels = np.argmax(Y, axis=1)
        return np.mean(predictions == true_labels) * 100


# ======================================================================
# 3. RUN IT
# ======================================================================

if __name__ == "__main__":
    X_train_raw, y_train_raw, X_test_raw, y_test_raw = load_mnist()

    X_train, Y_train = preprocess(X_train_raw, y_train_raw)
    X_test, Y_test = preprocess(X_test_raw, y_test_raw)

    print(f"X_train shape: {X_train.shape}")   # (60000, 784)
    print(f"Y_train shape: {Y_train.shape}")   # (60000, 10)

    net = SimpleMNISTNet(learning_rate=0.05, l2_strength=1e-4)
    net.train(X_train, Y_train, X_test, Y_test, epochs=20, batch_size=128)

X_train shape: (60000, 784)
Y_train shape: (60000, 10)
Epoch  1 | Loss 0.2800 | Train 91.96% | Test 92.22%
Epoch  2 | Loss 0.2100 | Train 94.07% | Test 94.25%
Epoch  3 | Loss 0.1691 | Train 95.18% | Test 95.11%
Epoch  4 | Loss 0.1420 | Train 95.95% | Test 95.56%
Epoch  5 | Loss 0.1263 | Train 96.33% | Test 96.04%
Epoch  6 | Loss 0.1080 | Train 96.93% | Test 96.49%
Epoch  7 | Loss 0.0970 | Train 97.27% | Test 96.70%
Epoch  8 | Loss 0.0877 | Train 97.54% | Test 96.93%
Epoch  9 | Loss 0.0779 | Train 97.81% | Test 97.10%
Epoch 10 | Loss 0.0720 | Train 97.98% | Test 97.14%
Epoch 11 | Loss 0.0649 | Train 98.25% | Test 97.23%
Epoch 12 | Loss 0.0585 | Train 98.41% | Test 97.40%
Epoch 13 | Loss 0.0552 | Train 98.51% | Test 97.46%
Epoch 14 | Loss 0.0527 | Train 98.55% | Test 97.48%
Epoch 15 | Loss 0.0460 | Train 98.85% | Test 97.64%
Epoch 16 | Loss 0.0453 | Train 98.80% | Test 97.58%
Epoch 17 | Loss 0.0396 | Train 99.04% | Test 97.70%
Epoch 18 | Loss 0.0352 | Train 99.14% | Test 97.81%
Epoch 19 